# VLM Consistency Test

Measures how stable Gemini responses are across N repeated calls for the three query types used in MAGPIE:

1. **Auto-detect** — given an image, does Gemini consistently name the object?
2. **Grasp strategy** — given image + name + PCA dims, does it consistently pick `symmetric / short_side / long_side`?
3. **DeliGrasp params** — given object name, how much do `mass_g`, `k`, `mu`, `aperture_mm` vary?

Run cells top to bottom. Camera does **not** need to be running — you can load any saved image.

In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────────────
import os, re, sys, time, statistics
from collections import Counter
from datetime import date
import numpy as np
import matplotlib.pyplot as plt
import cv2

# Load .env without python-dotenv
_env_path = os.path.expanduser('~/magpie_control/.env')
with open(_env_path) as _f:
    for _line in _f:
        _line = _line.strip()
        if _line and not _line.startswith('#') and '=' in _line:
            _k, _v = _line.split('=', 1)
            os.environ.setdefault(_k.strip(), _v.strip().strip('"').strip("'"))

api_key = os.environ.get('GEMINI_API_KEY', '')
assert api_key, 'GEMINI_API_KEY not set in .env'

from google import genai
from google.genai import types as gtypes

client = genai.Client(api_key=api_key)
MODEL  = 'gemini-2.5-flash'

# ── Config — edit these ────────────────────────────────────────────────────────
IMAGE_PATH  = '/tmp/test_obj.jpg'   # path to test image; see cell c02 to capture from camera
OBJECT_NAME = None                  # set to a string to skip auto-detect, e.g. 'red cube'
N           = 20                    # repetitions per test
MAJOR_MM    = 80                    # PCA major axis for strategy test (mm)
MINOR_MM    = 50                    # PCA minor axis for strategy test (mm)

print('Setup OK')

In [ ]:
# ── Load image ─────────────────────────────────────────────────────────────────
# Option A: load a saved file (set IMAGE_PATH in c01)
# Option B: capture live from ROS camera — uncomment the block below

# --- Option B: capture from camera (nodes must be running) ---
# import rclpy
# from sensor_msgs.msg import Image as RosImage
# NS = '/camera/gripper_camera/camera'
# _frame = None
# def _cb(msg):
#     global _frame
#     import numpy as np
#     _frame = np.frombuffer(msg.data, dtype=np.uint8).reshape(msg.height, msg.width, -1)
# if not rclpy.ok(): rclpy.init()
# _tmp_node = rclpy.create_node('img_capture')
# _sub = _tmp_node.create_subscription(RosImage, NS+'/color/image_raw', _cb, 1)
# import time
# for _ in range(30):
#     rclpy.spin_once(_tmp_node, timeout_sec=0.1)
#     if _frame is not None: break
# _tmp_node.destroy_node()
# img_rgb = _frame.copy()
# cv2.imwrite(IMAGE_PATH, cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR))
# print(f'Captured and saved to {IMAGE_PATH}')

# --- Option A ---
assert os.path.exists(IMAGE_PATH), (
    f'Image not found: {IMAGE_PATH}\n'
    'Either:\n'
    '  - Run Option B above to capture from camera, OR\n'
    '  - Save a jpg/png to that path and re-run this cell'
)

img_bgr = cv2.imread(IMAGE_PATH)
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

_, buf = cv2.imencode('.jpg', img_bgr)
img_bytes = buf.tobytes()

plt.figure(figsize=(6, 4))
plt.imshow(img_rgb)
plt.axis('off')
plt.title(IMAGE_PATH)
plt.tight_layout()
plt.show()
print(f'Image loaded: {img_rgb.shape[1]}x{img_rgb.shape[0]} px')

In [ ]:
# ── Helpers ─────────────────────────────────────────────────────────────────────

def img_part():
    return gtypes.Part.from_bytes(data=img_bytes, mime_type='image/jpeg')

def call_gemini(contents, system=None, delay=0.6):
    cfg = gtypes.GenerateContentConfig(system_instruction=system) if system else None
    for attempt in range(2):
        try:
            r = client.models.generate_content(model=MODEL, contents=contents, config=cfg)
            time.sleep(delay)
            return r.text.strip()
        except Exception as e:
            if attempt == 0:
                print(f'  [retry after error: {e}]')
                time.sleep(5)
            else:
                return f'ERROR: {e}'

DG_PROMPT = """Control a robot gripper with force control and contact information. \
The gripper's parameters can be adjusted corresponding to the type of object that it is trying \
to grasp as well as the kind of grasp it is attempting to perform.
The gripper has a measurable max force of 16N and min force of 0.15N, a maximum aperture of \
105mm and a minimum aperture of 1mm.

Some grasps may be incomplete, intended for observing force information about a given object.
Describe the grasp strategy using the following form:

[start of description]
* This {CHOICE: [is, is not]} a new grasp.
* In accordance with the user instruction, this grasp should be [GRASP_DESCRIPTION: <str>].
* This is a {CHOICE: [complete, incomplete]} grasp.
* This grasp {CHOICE: [does, does not]} contain multiple grasps.
* This grasp is for an object with {CHOICE: [high, medium, low]} weight.
* The object has an approximate mass of [PNUM: 0.0] grams
* This grasp is for an object with {CHOICE: [high, medium, low]} compliance.
* The object has an approximate spring constant of [PNUM: 0.0] Newtons per meter.
* The gripper and object have an approximate friction coefficient of [PNUM: 0.0]
* This grasp should set the goal aperture to [PNUM: 0.0] mm.
* If the gripper slips, this grasp should close an additional [PNUM: 0.0] mm.
* If the gripper slips, this grasp should increase the output force by [PNUM: 0.0] Newtons.
* [optional] Because of [GRASP_DESCRIPTION: <str>], this grasp sets the force to be \
{CHOICE: [lower, higher]} than the default minimum grasp force.
[end of description]

Rules:
1. Replace {PNUM: default_value} with a positive non-zero number.
2. Replace {CHOICE: [a, b, ...]} with one of the listed choices.
3. Replace [GRASP_DESCRIPTION: x] with a description of the grasp/object.
4. Spring constant: 20 N/m (very soft) to 2000 N/m (very stiff).
5. Force increase = max(0.05, k * additional_closure * 0.0001).
6. Always start with [start of description] and end with [end of description].
7. Give the full description. Do not skip non-optional points."""

def parse_dg(text):
    m = re.search(r'\[start of description\](.*?)\[end of description\]',
                  text, re.DOTALL | re.IGNORECASE)
    if not m:
        return None
    body = m.group(1)
    def _get(pattern):
        hit = re.search(pattern, body, re.IGNORECASE)
        return float(hit.group(1)) if hit else None
    return {
        'mass_g':      _get(r'approximate mass of ([0-9.]+) grams'),
        'k':           _get(r'spring constant of ([0-9.]+) Newtons per meter'),
        'mu':          _get(r'friction coefficient of ([0-9.]+)'),
        'aperture_mm': _get(r'goal aperture to ([0-9.]+) mm'),
    }

print('Helpers ready')

In [ ]:
# ── Test 1: Auto-detect ────────────────────────────────────────────────────────
detect_prompt = ('What is this object? Reply with ONLY a 2-4 word description, '
                 'nothing else. Example: "red rubber eraser"')

detect_results = []
for i in range(N):
    text = call_gemini([img_part(), detect_prompt])
    result = text.lower().strip('.').strip()
    detect_results.append(result)
    print(f'  {i+1:>2}/{N}: {result}')

detect_counts = Counter(detect_results)
detect_mode, detect_mode_count = detect_counts.most_common(1)[0]
detect_pct = detect_mode_count / N * 100
print(f'\nMost common: "{detect_mode}" ({detect_pct:.0f}% of calls)')

In [ ]:
# ── Test 2: Grasp strategy ─────────────────────────────────────────────────────
obj_name = OBJECT_NAME or detect_mode
ratio = MAJOR_MM / max(MINOR_MM, 1)
print(f'Object: "{obj_name}"  major={MAJOR_MM}mm  minor={MINOR_MM}mm  ratio={ratio:.2f}')

strategy_prompt = (
    f'Object: "{obj_name}"\n'
    f'Point cloud extents: major={MAJOR_MM:.0f}mm, minor={MINOR_MM:.0f}mm (ratio={ratio:.2f})\n\n'
    f'Choose the best gripper strategy — reply with EXACTLY one word:\n'
    f'  symmetric  — shape is round/square, any angle works (cube, ball, cylinder)\n'
    f'  short_side — grip perpendicular to longest dimension (box, book, phone)\n'
    f'  long_side  — grip parallel to longest dimension (pen, banana, screwdriver)\n'
)

strategy_results = []
for i in range(N):
    text = call_gemini([img_part(), strategy_prompt])
    word = text.lower().split()[0] if text else 'parse_error'
    if word not in ('symmetric', 'short_side', 'long_side'):
        word = f'other:{word}'
    strategy_results.append(word)
    print(f'  {i+1:>2}/{N}: {word}')

strategy_counts = Counter(strategy_results)
strat_mode, strat_mode_count = strategy_counts.most_common(1)[0]
strat_pct = strat_mode_count / N * 100
print(f'\nMost common: {strat_mode} ({strat_pct:.0f}% of calls)')

In [ ]:
# ── Test 3: DeliGrasp params ───────────────────────────────────────────────────
print(f'Object: "{obj_name}"  ({N} calls, ~{N*1.5:.0f}s)')

dg_results = []
for i in range(N):
    text = call_gemini(f'Pick up the {obj_name}.', system=DG_PROMPT, delay=1.0)
    parsed = parse_dg(text)
    if parsed and all(v is not None for v in parsed.values()):
        dg_results.append(parsed)
        print(f'  {i+1:>2}/{N}: mass={parsed["mass_g"]:>6.1f}g  '
              f'k={parsed["k"]:>6.1f}N/m  mu={parsed["mu"]:.2f}  '
              f'ap={parsed["aperture_mm"]:>5.1f}mm')
    else:
        print(f'  {i+1:>2}/{N}: PARSE FAILED')

print(f'\nParsed OK: {len(dg_results)}/{N}')

In [ ]:
# ── Plots ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Plot 1: auto-detect distribution
ax = axes[0]
labels, counts = zip(*detect_counts.most_common())
ax.barh(labels, counts, color='steelblue')
ax.set_xlabel('Count')
ax.set_title(f'Auto-detect  (N={N})')
ax.invert_yaxis()

# Plot 2: strategy distribution
ax = axes[1]
all_strategies = ['symmetric', 'short_side', 'long_side']
strat_vals = [strategy_counts.get(s, 0) for s in all_strategies]
colors = ['#4CAF50' if s == strat_mode else '#90CAF9' for s in all_strategies]
ax.bar(all_strategies, strat_vals, color=colors)
ax.set_ylabel('Count')
ax.set_title(f'Grasp strategy  (N={N})')
ax.set_ylim(0, N)

# Plot 3: DeliGrasp param spread (CV)
ax = axes[2]
if dg_results:
    fields = ['mass_g', 'k', 'mu', 'aperture_mm']
    cvs = []
    for f in fields:
        vals = [r[f] for r in dg_results]
        mu_v = statistics.mean(vals)
        sd_v = statistics.stdev(vals) if len(vals) > 1 else 0.0
        cvs.append((sd_v / mu_v * 100) if mu_v else 0)
    bar_colors = ['#4CAF50' if cv < 20 else '#FF9800' if cv < 50 else '#F44336' for cv in cvs]
    ax.bar(fields, cvs, color=bar_colors)
    ax.axhline(20, color='gray', linestyle='--', linewidth=0.8, label='20% CV')
    ax.set_ylabel('CV (%)')
    ax.set_title(f'DeliGrasp param variability  (N={len(dg_results)})')
    ax.legend()
else:
    ax.text(0.5, 0.5, 'No parsed results', ha='center', va='center', transform=ax.transAxes)

fig.suptitle(f'VLM Consistency — "{obj_name}"  [{date.today()}]', fontsize=13)
plt.tight_layout()
plot_path = f'/tmp/vlm_consistency_{date.today()}.png'
plt.savefig(plot_path, dpi=120)
plt.show()
print(f'Plot saved to {plot_path}')

In [ ]:
# ── Save results to markdown ───────────────────────────────────────────────────
out_path = os.path.expanduser(
    f'~/magpie_control/tests/vlm_consistency_{date.today()}.md')

lines = [
    '# VLM Consistency Test\n',
    f'**Date:** {date.today()}  ',
    f'**Image:** `{IMAGE_PATH}`  ',
    f'**Object:** {obj_name}  ',
    f'**Model:** {MODEL}  ',
    f'**N per test:** {N}  \n',
    '## Test 1 — Auto-detect\n',
    '| Response | Count | % |',
    '|---|---|---|',
]
for name, cnt in detect_counts.most_common():
    lines.append(f'| {name} | {cnt} | {cnt/N*100:.0f}% |')
lines.append(f'\n**Consistency: {detect_pct:.0f}% on most common answer "{detect_mode}"**\n')

lines += [
    '## Test 2 — Grasp strategy\n',
    '| Strategy | Count | % |',
    '|---|---|---|',
]
for s, cnt in strategy_counts.most_common():
    lines.append(f'| {s} | {cnt} | {cnt/N*100:.0f}% |')
lines.append(f'\n**Consistency: {strat_pct:.0f}% on `{strat_mode}`**\n')

lines += [
    f'## Test 3 — DeliGrasp params  (parsed {len(dg_results)}/{N})\n',
    '| Parameter | Mean | Std Dev | CV (%) |',
    '|---|---|---|---|',
]
units = {'mass_g': 'g', 'k': 'N/m', 'mu': '', 'aperture_mm': 'mm'}
for field in ('mass_g', 'k', 'mu', 'aperture_mm'):
    if dg_results:
        vals = [r[field] for r in dg_results]
        mu_v = statistics.mean(vals)
        sd_v = statistics.stdev(vals) if len(vals) > 1 else 0.0
        cv   = (sd_v / mu_v * 100) if mu_v else 0
        lines.append(f'| {field} ({units[field]}) | {mu_v:.2f} | {sd_v:.2f} | {cv:.1f}% |')
lines.append('\n*CV = coefficient of variation; green <20%, orange <50%, red ≥50%*')

with open(out_path, 'w') as f:
    f.write('\n'.join(lines))

print(f'Saved to {out_path}')
print('\n=== SUMMARY ===')
print(f'  Auto-detect:    {detect_pct:.0f}% consistent  (mode: "{detect_mode}")')
print(f'  Grasp strategy: {strat_pct:.0f}% consistent  (mode: {strat_mode})')
if dg_results:
    for field in ('mass_g', 'k', 'mu', 'aperture_mm'):
        vals = [r[field] for r in dg_results]
        mu_v = statistics.mean(vals)
        sd_v = statistics.stdev(vals) if len(vals) > 1 else 0.0
        cv   = (sd_v / mu_v * 100) if mu_v else 0
        flag = '✓' if cv < 20 else '~' if cv < 50 else '!'
        print(f'  {field:<12}  mean={mu_v:>8.2f}  CV={cv:>5.1f}%  {flag}')